In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
carpeta_actual = Path.cwd()

if carpeta_actual.name == "notebooks":
    carpeta_proyecto = carpeta_actual.parent
else:
    carpeta_proyecto = carpeta_actual

carpeta_datos = carpeta_proyecto / "datos"
carpeta_procesados = carpeta_proyecto / "datos_procesados"
carpeta_entregas = carpeta_proyecto / "entregas"

carpeta_entregas.mkdir(
    parents=True,
    exist_ok=True
)

ruta_datos = (
    carpeta_procesados
    / "solicitudes_train_procesado.csv"
)

solicitudes_modelo = pd.read_csv(ruta_datos)

print("Archivo cargado desde:", ruta_datos)
print("Filas y columnas:", solicitudes_modelo.shape)
solicitudes_modelo.head()

In [ ]:
solicitudes_modelo["Fecha_Solicitud"] = pd.to_datetime(
    solicitudes_modelo["Fecha_Solicitud"]
)

In [ ]:
fecha_corte = "2025-10-01"

datos_entrenamiento = solicitudes_modelo[
    solicitudes_modelo["Fecha_Solicitud"] < fecha_corte
].copy()

datos_validacion = solicitudes_modelo[
    solicitudes_modelo["Fecha_Solicitud"] >= fecha_corte
].copy()

print("Entrenamiento:", datos_entrenamiento.shape)
print("Validación:", datos_validacion.shape)

Entrenamiento: (24391, 12)
Validación: (8025, 12)


In [ ]:
columnas_modelo = [
    "Monto_Solicitado",
    "Tasa_Interes",
    "Ingreso_Mensual",
    "meses_historial",
    "saldo_historial",
    "atraso_prom_hist",
    "atraso_max"
]

In [ ]:
X_entrenamiento = datos_entrenamiento[columnas_modelo]
y_entrenamiento = datos_entrenamiento["Estado_Final"]

X_validacion = datos_validacion[columnas_modelo]
y_validacion = datos_validacion["Estado_Final"]

print("X entrenamiento:", X_entrenamiento.shape)
print("y entrenamiento:", y_entrenamiento.shape)
print("X validación:", X_validacion.shape)
print("y validación:", y_validacion.shape)

X entrenamiento: (24391, 7)
y entrenamiento: (24391,)
X validación: (8025, 7)
y validación: (8025,)


In [ ]:
X_entrenamiento.head()


,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,meses_historial,saldo_historial,atraso_prom_hist,atraso_max
0,7900,0.2333,4461,14,3695.225714,4.857143,25
2,8600,0.2368,4035,15,178.574667,20.733333,57
3,13300,0.2826,4018,18,7589.356667,0.944444,4
4,6600,0.2314,3357,12,3888.616667,3.750000,32
6,10700,0.2493,3952,11,42.991818,25.727273,63


In [ ]:
print("Entrenamiento:")
print((y_entrenamiento.value_counts(normalize=True).sort_index() * 100).round(2))

print("\nValidación:")
print((y_validacion.value_counts(normalize=True).sort_index() * 100).round(2))

Entrenamiento:
Estado_Final
0    16.02
1    83.98
Name: proportion, dtype: float64

Validación:
Estado_Final
0    15.74
1    84.26
Name: proportion, dtype: float64


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
#from sklearn.linear_model import LogisticRegression
#from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
#from sklearn.ensemble import GradientBoostingClassifier

modelo_base = Pipeline([
    ("escalado", StandardScaler()),
    ("modelo", DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=600,
        random_state=42
    ))
])

modelo_base.fit(X_entrenamiento, y_entrenamiento)

print("Modelo entrenado")

Modelo entrenado


In [ ]:
probabilidad_pago = modelo_base.predict_proba(X_validacion)[:, 1]

resultados_validacion = datos_validacion[
    [
        "ID_Solicitud",
        "Monto_Solicitado",
        "Tasa_Interes",
        "Estado_Final"
    ]
].copy()

resultados_validacion["Probabilidad_Pago"] = probabilidad_pago

resultados_validacion.head()

,ID_Solicitud,Monto_Solicitado,Tasa_Interes,Estado_Final,Probabilidad_Pago
1,2,17400,0.2507,1,0.936334
5,6,7100,0.2663,1,0.974777
11,12,22500,0.2964,1,0.930233
13,14,12700,0.2551,0,0.974777
15,16,10700,0.2531,1,0.996548


In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_validacion,
    probabilidad_pago
)

print("ROC-AUC:", round(auc, 4))

ROC-AUC: 0.8266


In [ ]:
resultados_validacion["Ganancia_Esperada"] = (
    resultados_validacion["Probabilidad_Pago"]
    * resultados_validacion["Monto_Solicitado"]
    * resultados_validacion["Tasa_Interes"]
    +
    (1 - resultados_validacion["Probabilidad_Pago"])
    * (-resultados_validacion["Monto_Solicitado"])
)

resultados_validacion[
    [
        "ID_Solicitud",
        "Monto_Solicitado",
        "Tasa_Interes",
        "Probabilidad_Pago",
        "Ganancia_Esperada"
    ]
].head()

,ID_Solicitud,Monto_Solicitado,Tasa_Interes,Probabilidad_Pago,Ganancia_Esperada
1,2,17400,0.2507,0.936334,2976.662503
5,6,7100,0.2663,0.974777,1663.960846
11,12,22500,0.2964,0.930233,4633.953488
13,14,12700,0.2551,0.974777,2837.728323
15,16,10700,0.2531,0.996548,2661.881726


In [ ]:
presupuesto = 2_000_000

resultados_validacion["Rentabilidad_Esperada"] = (
    resultados_validacion["Ganancia_Esperada"]
    / resultados_validacion["Monto_Solicitado"]
)

resultados_validacion["Ganancia_Real"] = np.where(
    resultados_validacion["Estado_Final"] == 1,
    resultados_validacion["Monto_Solicitado"]
    * resultados_validacion["Tasa_Interes"],
    -resultados_validacion["Monto_Solicitado"]
)

candidatos = (
    resultados_validacion[
        resultados_validacion["Ganancia_Esperada"] > 0
    ]
    .sort_values("Rentabilidad_Esperada", ascending=False)
    .copy()
)

candidatos["Monto_Acumulado"] = (
    candidatos["Monto_Solicitado"].cumsum()
)

cartera = candidatos[
    candidatos["Monto_Acumulado"] <= presupuesto
].copy()

print("Préstamos aprobados:", len(cartera))
print("Monto invertido:", cartera["Monto_Solicitado"].sum())
print("Ganancia esperada:", cartera["Ganancia_Esperada"].sum())
print("Ganancia real:", cartera["Ganancia_Real"].sum())

Préstamos aprobados: 87
Monto invertido: 1997100
Ganancia esperada: 583352.2928814633
Ganancia real: 579917.78


In [ ]:
print(cartera["Estado_Final"].value_counts())

ganancia_si_todos_pagaran = (
    cartera["Monto_Solicitado"] * cartera["Tasa_Interes"]
).sum()

print("Ganancia si todos pagaran:", ganancia_si_todos_pagaran)
print("Ganancia real:", cartera["Ganancia_Real"].sum())

Estado_Final
1    85
0     2
Name: count, dtype: int64
Ganancia si todos pagaran: 607941.61
Ganancia real: 579917.78


In [ ]:
ruta_test = (
    carpeta_datos
    / "solicitudes_test.csv"
)

ruta_banco = (
    carpeta_datos
    / "comportamiento_bancario.csv"
)

solicitudes_test = pd.read_csv(ruta_test)
comportamiento = pd.read_csv(ruta_banco)

solicitudes_test["Fecha_Solicitud"] = pd.to_datetime(
    solicitudes_test["Fecha_Solicitud"]
)

comportamiento["Mes_Referencia"] = pd.to_datetime(
    comportamiento["Mes_Referencia"]
)

print("Test:", solicitudes_test.shape)
solicitudes_test.head()

Test: (3061, 7)


,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud
0,32417,27074,2300,0.2771,2596,Consumo,2026-01-01
1,32418,28172,10900,0.3291,6905,Consumo,2026-01-01
2,32419,27547,1300,0.2485,3707,Educacion,2026-01-01
3,32420,27689,1600,0.2699,3314,Consumo,2026-01-01
4,32421,26219,900,0.2317,3870,Educacion,2026-01-01


In [ ]:
historial_test = solicitudes_test[
    ["ID_Solicitud", "ID_Usuario", "Fecha_Solicitud"]
].merge(
    comportamiento,
    on="ID_Usuario",
    how="left"
)

historial_test = historial_test[
    historial_test["Mes_Referencia"]
    < historial_test["Fecha_Solicitud"]
].copy()

resumen_test = (
    historial_test
    .groupby("ID_Solicitud")
    .agg(
        meses_historial=("Mes_Referencia", "count"),
        saldo_historial=("Saldo_Promedio_Mensual", "mean"),
        atraso_prom_hist=("Dias_Atraso_Otras_Deudas", "mean"),
        atraso_max=("Dias_Atraso_Otras_Deudas", "max")
    )
    .reset_index()
)

test_modelo = solicitudes_test.merge(
    resumen_test,
    on="ID_Solicitud",
    how="left"
)

print("Tamaño:", test_modelo.shape)
print(test_modelo[columnas_modelo].isna().sum())

Tamaño: (3061, 11)
Monto_Solicitado    0
Tasa_Interes        0
Ingreso_Mensual     0
meses_historial     0
saldo_historial     0
atraso_prom_hist    0
atraso_max          0
dtype: int64


In [ ]:
from sklearn.base import clone

X_total = solicitudes_modelo[columnas_modelo]
y_total = solicitudes_modelo["Estado_Final"]
X_test = test_modelo[columnas_modelo]

modelo_final = clone(modelo_base)
modelo_final.fit(X_total, y_total)

probabilidad_pago_test = modelo_final.predict_proba(X_test)[:, 1]

print("Predicciones realizadas:", len(probabilidad_pago_test))

Predicciones realizadas: 3061


In [ ]:
resultados_test = test_modelo[
    ["ID_Solicitud", "Monto_Solicitado", "Tasa_Interes"]
].copy()

resultados_test["Probabilidad_Pago"] = probabilidad_pago_test

resultados_test["Ganancia_Esperada"] = (
    resultados_test["Probabilidad_Pago"]
    * resultados_test["Monto_Solicitado"]
    * resultados_test["Tasa_Interes"]
    +
    (1 - resultados_test["Probabilidad_Pago"])
    * (-resultados_test["Monto_Solicitado"])
)

resultados_test["Rentabilidad_Esperada"] = (
    resultados_test["Ganancia_Esperada"]
    / resultados_test["Monto_Solicitado"]
)

candidatos_test = (
    resultados_test[
        resultados_test["Ganancia_Esperada"] > 0
    ]
    .sort_values("Rentabilidad_Esperada", ascending=False)
    .copy()
)

candidatos_test.head()

,ID_Solicitud,Monto_Solicitado,Tasa_Interes,Probabilidad_Pago,Ganancia_Esperada,Rentabilidad_Esperada
3031,35448,4700,0.3208,0.996377,1485.268116,0.316014
57,32474,5200,0.3201,0.996377,1639.648551,0.315317
2487,34904,4300,0.3224,0.994137,1352.983116,0.314647
730,33147,8800,0.3168,0.996377,2745.855072,0.312029
2701,35118,4100,0.3222,0.990917,1271.782761,0.310191


In [ ]:
presupuesto = 2_000_000

seleccionados = []
monto_acumulado = 0

for indice, prestamo in candidatos_test.iterrows():
    nuevo_total = (
        monto_acumulado + prestamo["Monto_Solicitado"]
    )

    if nuevo_total <= presupuesto:
        seleccionados.append(indice)
        monto_acumulado = nuevo_total

cartera_test = candidatos_test.loc[seleccionados].copy()

print("Préstamos aprobados:", len(cartera_test))
print("Monto aprobado:", cartera_test["Monto_Solicitado"].sum())
print("Ganancia esperada:", cartera_test["Ganancia_Esperada"].sum())

Préstamos aprobados: 687
Monto aprobado: 2000000
Ganancia esperada: 520514.92598210654


In [ ]:
entrega = solicitudes_test[["ID_Solicitud"]].copy()

entrega["Decision_Aprobacion"] = (
    entrega["ID_Solicitud"]
    .isin(cartera_test["ID_Solicitud"])
    .astype(int)
)

verificacion = solicitudes_test.merge(
    entrega,
    on="ID_Solicitud",
    how="left"
)

monto_verificado = verificacion.loc[
    verificacion["Decision_Aprobacion"] == 1,
    "Monto_Solicitado"
].sum()

print("Filas de entrega:", len(entrega))
print(entrega["Decision_Aprobacion"].value_counts())
print("IDs duplicados:", entrega["ID_Solicitud"].duplicated().sum())
print("Monto aprobado:", monto_verificado)

Filas de entrega: 3061
Decision_Aprobacion
0    2374
1     687
Name: count, dtype: int64
IDs duplicados: 0
Monto aprobado: 2000000


In [ ]:
ruta_entrega = (
    carpeta_entregas
    / "cartera_entrega_final.csv"
)

entrega.to_csv(
    ruta_entrega,
    index=False
)

print("Archivo guardado en:", ruta_entrega)
entrega.head()

Archivo guardado en: /content/drive/MyDrive/TP_IA_LaDobleS/entregas/cartera.csv


,ID_Solicitud,Decision_Aprobacion
0,32417,0
1,32418,0
2,32419,0
3,32420,1
4,32421,0
